# 2.0 Refactorización de variables

Paso intermedio entre la exploración (`1.0-p2-exploracion-mibici`) y `features.py`. Cada bloque de la exploración se convierte en una función, se revisa contra los datos y al final se compara con el módulo.

In [1]:
import hashlib
import json

import numpy as np
import pandas as pd

from mibici_equipo39 import config as C

departures = pd.read_parquet(C.INTERIM / "departures", columns=["station_id", "timestamp"])
arrivals = pd.read_parquet(C.INTERIM / "arrivals", columns=["station_id", "timestamp"])
stations = pd.read_parquet(C.PROCESSED / "stations.parquet")
len(departures), len(arrivals), len(stations)

(4497119, 4497119, 484)

## Eventos

In [2]:
def make_events(clean_departures, clean_arrivals):
    departures = clean_departures[["station_id", "timestamp"]].assign(event=-1)
    arrivals = clean_arrivals[["station_id", "timestamp"]].assign(event=1)
    events = pd.concat([departures, arrivals], ignore_index=True)
    events["station_id"] = events["station_id"].astype("string")
    return events.sort_values(["timestamp", "station_id"]).reset_index(drop=True)

events = make_events(departures, arrivals)
events["event"].value_counts()

event
-1    4497119
 1    4497119
Name: count, dtype: int64

## Malla estación-hora

In [3]:
def aggregate_hourly(events, stations, start=C.SOURCE_START, end=C.SOURCE_END_EXCLUSIVE):
    df = events.loc[events["station_id"].isin(stations["station_id"])].copy()
    df["hour"] = df["timestamp"].dt.floor("h")
    df = df.loc[df["hour"].ge(start) & df["hour"].lt(end)]
    counts = (df.groupby(["station_id", "hour", "event"], observed=True).size()
              .unstack("event", fill_value=0).reindex(columns=[-1, 1], fill_value=0)
              .rename(columns={-1: "departures", 1: "arrivals"}))
    index = pd.MultiIndex.from_product(
        [sorted(df["station_id"].unique()), pd.date_range(start, end, freq="h", inclusive="left")],
        names=["station_id", "hour"])
    hourly = counts.reindex(index, fill_value=0).reset_index()
    hourly.columns.name = None
    hourly["station_id"] = hourly["station_id"].astype("string")
    return hourly

hourly = aggregate_hourly(events, stations)
n_hours = hourly["hour"].nunique()
print(hourly["station_id"].nunique(), "estaciones x", n_hours, "horas =", len(hourly))
print("horas sin salidas:", round(hourly["departures"].eq(0).mean(), 3))
print("llegadas después del rango que se descartan:", int((arrivals["timestamp"] >= C.SOURCE_END_EXCLUSIVE).sum()))

469 estaciones x 8760 horas = 4108440
horas sin salidas: 0.621
llegadas después del rango que se descartan: 16


> **Por qué la malla completa.** Más de la mitad de las horas no tiene salidas. Si solo se agregan las horas con viajes, esos ceros desaparecen, el modelo sobreestima la demanda y `shift(24)` ya no apunta a la misma hora del día anterior. El rango queda fijo en `config.py` para que las llegadas que caen después del 30 de septiembre no agreguen horas de más.

## Calendario

Se conservan las variables de la exploración de Stefania: hora, día, mes, fin de semana y su versión cíclica.

In [4]:
def add_calendar_features(df):
    df = df.copy()
    df["hour_of_day"] = df["hour"].dt.hour
    df["day_of_week"] = df["hour"].dt.dayofweek
    df["month"] = df["hour"].dt.month
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour_of_day"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour_of_day"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df

hourly = add_calendar_features(hourly)

## Rezagos

Antes de programarlos se revisa si la demanda pasada sirve para explicar la siguiente hora.

In [5]:
ordered = hourly.sort_values(["station_id", "hour"]).reset_index(drop=True)
by_station = ordered.groupby("station_id", sort=False, observed=True)["departures"]
autocorr = {f"lag_{k}": ordered["departures"].corr(by_station.shift(k)) for k in [1, 2, 24, 168]}
calendar_only = ordered.groupby(["hour_of_day", "day_of_week"])["departures"].transform("mean")
autocorr["media_calendario_sin_estacion"] = ordered["departures"].corr(calendar_only)
pd.Series(autocorr).round(3)

lag_1                            0.682
lag_2                            0.564
lag_24                           0.646
lag_168                          0.704
media_calendario_sin_estacion    0.342
dtype: float64

> **Por qué rezagos.** La correlación de las salidas con su valor de hace 1, 24 y 168 horas es mayor que la de una media que solo usa el calendario. El calendario dice cuándo hay pico, pero no en qué estación: dos estaciones con la misma hora y día pueden tener 0 o 20 salidas. Los rezagos traen ese nivel propio de cada estación sin codificar 469 identificadores.

In [6]:
LAGS, WINDOWS = (1, 2, 24, 168), (24, 168)

def add_lagged_features(df):
    df = df.sort_values(["station_id", "hour"]).reset_index(drop=True)
    by_station = df.groupby("station_id", sort=False, observed=True)

    def rolling(series, window, how):
        grouped = series.groupby(df["station_id"], sort=False, observed=True).rolling(window, min_periods=window)
        return getattr(grouped, how)().reset_index(level=0, drop=True).sort_index()

    net = df["arrivals"] - df["departures"]
    new = {}
    for column, prefix in [("departures", "dep"), ("arrivals", "arr")]:
        for lag in LAGS:
            new[f"{prefix}_lag_{lag}"] = by_station[column].shift(lag)
        past = by_station[column].shift(1)
        for window in WINDOWS:
            new[f"{prefix}_mean_{window}"] = rolling(past, window, "mean")
    past_net = net.groupby(df["station_id"], sort=False, observed=True).shift(1)
    new["net_lag_1"] = past_net
    new["net_sum_24"] = rolling(past_net, 24, "sum")
    return pd.concat([df, pd.DataFrame(new).astype("float32")], axis=1)

lagged = add_lagged_features(hourly)
row = lagged.loc[lagged["station_id"].eq(lagged["station_id"].iloc[0])].iloc[500]
week_before = lagged.set_index(["station_id", "hour"]).loc[(row["station_id"], row["hour"] - pd.Timedelta("168h")), "departures"]
print("dep_lag_168 = salidas de la semana anterior:", row["dep_lag_168"] == week_before)
past_24 = lagged.set_index(["station_id", "hour"]).loc[row["station_id"]].loc[row["hour"] - pd.Timedelta("24h"): row["hour"] - pd.Timedelta("1h"), "departures"]
print("dep_mean_24 no incluye la hora t:", np.isclose(row["dep_mean_24"], past_24.mean()))

dep_lag_168 = salidas de la semana anterior: True
dep_mean_24 no incluye la hora t: True


> **Sin fuga.** `shift(1)` va antes de `rolling`, así que la media de 24 horas termina en t-1. Se descartan las filas anteriores al 9 de octubre (`FEATURES_START`): el 1 de octubre lo excluye la ingesta y los siete días siguientes no tienen rezago de 168 horas completo.

In [7]:
lagged = lagged.loc[lagged["hour"].ge(C.FEATURES_START)].reset_index(drop=True)
lagged.filter(like="_lag_").isna().sum().sum()

np.int64(0)

## Partición

In [8]:
same_day = ordered.assign(next_hour=by_station.shift(-1)).dropna()
print("correlación entre una hora y la siguiente:", round(same_day["departures"].corr(same_day["next_hour"]), 3))

correlación entre una hora y la siguiente: 0.682


> **Por qué días completos al azar.** Las horas seguidas de un mismo día están muy correlacionadas. Si las filas estación-hora se repartieran al azar, el modelo entrenaría con las 8:00 y las 10:00 y se evaluaría con las 9:00 del mismo día, que ya casi conoce, y la métrica saldría optimista. Por eso cada día completo va a un solo conjunto. El sorteo es por mes para que los doce meses, incluido el pico de septiembre, aparezcan en train, validación y prueba. El `split_id` es el hash del mapa de días: con la misma semilla sale el mismo id, cosa que no pasaba con el `uuid4` original.

In [9]:
def random_grouped_split(df, seed=C.SEED):
    df = df.copy()
    df["date"] = df["hour"].dt.normalize()
    rng = np.random.default_rng(seed)
    assignments = {}
    for _, days in df.groupby(df["date"].dt.to_period("M"))["date"]:
        days = np.sort(days.unique())
        rng.shuffle(days)
        cut_train = int(np.floor(C.TRAIN_SIZE * len(days) + 0.5))
        cut_val = int(np.floor((C.TRAIN_SIZE + C.VAL_SIZE) * len(days) + 0.5))
        for i, day in enumerate(days):
            assignments[pd.Timestamp(day)] = "train" if i < cut_train else "validation" if i < cut_val else "test"
    df["split"] = df["date"].map(assignments)
    days = {s: sorted(d.strftime("%Y-%m-%d") for d, v in assignments.items() if v == s) for s in ["train", "validation", "test"]}
    split_id = hashlib.sha256(json.dumps({"seed": seed, "days": days}, sort_keys=True).encode()).hexdigest()
    return df.drop(columns="date"), split_id, days

split_df, split_id, days = random_grouped_split(lagged)
_, again, _ = random_grouped_split(lagged)
print("mismo id en dos corridas:", split_id == again)
pd.Series({k: len(v) for k, v in days.items()}).rename("días").to_frame().assign(proporción=lambda d: (d["días"] / d["días"].sum()).round(3))

mismo id en dos corridas: True


,días,proporción
train,252,0.706
validation,52,0.146
test,53,0.148


> **Por qué se redondean los cortes acumulados.** Truncar train y validación por mes dejaba 69/13/18 porque todo el sobrante caía en prueba. Redondear el corte de 70% y el de 85% deja los tres conjuntos cerca de 70/15/15.

## Comparación con el módulo

Las funciones pasan a `mibici_equipo39/features.py` con la misma lógica; el módulo además lee rutas y proporciones de `config.py` y escribe `split_manifest.json`.

In [10]:
from mibici_equipo39 import features as F

module = F.add_lagged_features(F.add_calendar_features(F.aggregate_hourly(F.make_events(departures, arrivals), stations)))
module = module.loc[module["hour"].ge(C.FEATURES_START)].reset_index(drop=True)
module, manifest = F.random_grouped_split(module, data_version="notebook")
cols = ["station_id", "hour", "departures", "arrivals", "split", *F.FEATURE_COLUMNS]
pd.testing.assert_frame_equal(split_df[cols].reset_index(drop=True), module[cols].reset_index(drop=True), check_dtype=False)
print("notebook y módulo coinciden en", len(module), "filas")

notebook y módulo coinciden en 4018392 filas
